# 07 · Provider, capability e costi

Mini-harness provider-neutral. Solo Python standard library: nessuna API, chiave o
import dal progetto. Costruiamo contratti, preflight, costo e tassonomia errori.

## Obiettivi, prerequisiti e modalità di lettura

Costruirai un contratto provider-neutral con capability, costi ed errori. Durata: 20–30 minuti. Tutto gira offline con standard library.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## 1 · Descrivere capacità, non indovinarle

Un nome modello non garantisce tool, structured output o caching. Le capacità sono
dati espliciti usati prima del run.

### Spiegazione del blocco · Contratti del modello

Le dataclass distinguono identità, capacità e prezzi. Due provider che espongono la stessa API possono comunque dichiarare feature diverse.

In [ ]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True)
class Capabilities:
    context_window: int
    tools: bool
    structured_output: bool
    parallel_tools: bool

@dataclass(frozen=True)
class Model:
    provider: str
    name: str
    capabilities: Capabilities
    input_per_million: Decimal
    output_per_million: Decimal

cloud = Model("cloud", "smart", Capabilities(128_000, True, True, True), Decimal("1"), Decimal("6"))
local = Model("local", "small", Capabilities(8_192, True, False, False), Decimal("0"), Decimal("0"))

### Output atteso

Nessun output. Sono disponibili profili `cloud` e `local`.

## 2 · Capability gate prima del lavoro

### Spiegazione del blocco · Capability preflight

Il controllo rifiuta richieste incompatibili prima di spendere token o avviare lavoro parziale.

In [ ]:
def preflight(model: Model, *, needs_structured: bool, input_tokens: int) -> None:
    if input_tokens >= model.capabilities.context_window:
        raise ValueError("context_length")
    if needs_structured and not model.capabilities.structured_output:
        raise ValueError("unsupported_structured_output")

preflight(cloud, needs_structured=True, input_tokens=2_000)
try:
    preflight(local, needs_structured=True, input_tokens=2_000)
except ValueError as error:
    print("Rifiuto anticipato:", error)

### Output atteso

Il cloud passa; il locale stampa `Rifiuto anticipato: unsupported_structured_output`.

## 3 · Usage normalizzato e costo confrontabile

### Spiegazione del blocco · Costo normalizzato

La stessa formula converte token in costo per qualunque provider. Un modello locale mantiene costo API zero senza perdere le metriche di utilizzo.

In [ ]:
def cost(model: Model, input_tokens: int, output_tokens: int) -> Decimal:
    million = Decimal(1_000_000)
    return (Decimal(input_tokens) * model.input_per_million + Decimal(output_tokens) * model.output_per_million) / million

for candidate in (cloud, local):
    print(candidate.provider, cost(candidate, 10_000, 2_000))

### Output atteso

Una riga per provider: costo cloud positivo e costo local pari a `0`.

## 4 · Errori vendor-specifici → decisioni comuni

### Spiegazione del blocco · Tassonomia degli errori

Messaggi diversi vengono ridotti a categorie operative e flag retryable. Il runner può reagire senza conoscere ogni SDK.

In [ ]:
def classify_error(message: str) -> tuple[str, bool]:
    text = message.casefold()
    if "429" in text or "rate limit" in text:
        return "rate_limit", True
    if "timeout" in text:
        return "timeout", True
    if "401" in text or "api key" in text:
        return "auth", False
    if "context" in text:
        return "context_length", False
    return "unknown", False

assert classify_error("HTTP 429 rate limit") == ("rate_limit", True)
assert classify_error("401 invalid API key") == ("auth", False)
print("Tassonomia verificata")

### Output atteso

`Tassonomia verificata`; gli assert devono passare.

## Prova tu

Aggiungi un provider con tool ma senza tool paralleli. Fai fallire il preflight solo
quando il piano contiene due chiamate indipendenti da eseguire insieme.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: matrice delle capability

### Spiegazione del blocco

Confrontare profili in tabella rende evidente quali task richiedono fallback o escalation.

In [ ]:
for candidato in (cloud, local):
    cap = candidato.capabilities
    print(candidato.name, {"tools": cap.tools, "structured": cap.structured_output, "parallel": cap.parallel_tools})

### Output atteso

Il profilo cloud mostra tutte le capability vere; quello locale conserva structured output e parallel tools disabilitati.

## Esempio aggiuntivo: policy di retry

### Spiegazione del blocco

La tassonomia diventa utile quando guida una decisione operativa coerente.

In [ ]:
for messaggio in ("HTTP 429 rate limit", "request timeout", "401 invalid API key", "maximum context reached"):
    categoria, retry = classify_error(messaggio)
    azione = "retry con backoff" if retry else "stop o correzione configurazione"
    print(categoria, "->", azione)

### Output atteso

Rate limit e timeout producono retry; auth e context length producono stop/correzione.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.